# 1. 사이킷런(Scikit-learn)이란?
- 파이썬을 위한 가장 인기 있는 머신러닝 라이브러리 중 하나
- 복잡한 머신러닝 모델을 처음부터 직접 만드는 대신, 이미 잘 만들어진 모델을 포함한 각종 도구들을 제공

## 1-1. 사이킷런 사용 3단계 파이프라인
- 대부분의 모듈이 규칙에 따라 작동.
- 아래 3단계를 익히면, 어떤 기능이든 대부분 적용하여 활용 가능

1. Instance 선언
    - 데이터를 표준화 하고 싶다? `StandardScaler` 선언
    ```python
    scaler = StandardScaler
    ```

2. `.fit()`
    - 데이터에 맞게 조정하는 단계
    - 어떤 데이터를 기준으로 작동해야 할지 적용
    - 예를들어, `StandardScaler`는 `.fit()`을 통해 데이터의 평균과 표준편차를 계산하고 기억

3. `.transform()` 또는 `.predict()`
    - 직접 사용하는 단계
    - `.fit()`을 통해 학습된 규칙 (평균, 표준편차 등)을 바탕으로 실제 데이터를 변환하거나 예측


# 2. 실습: 데이터 전처리
> `scikit-learn` 라이브러리를 사용해 데이터를 모델링에 맞게 전처리 

## 2-1. 들어가기 전에
- 모델링 시작 전, 반드시 데이터를 다듬을 필요가 있음.
- 데이터 전처리를 올바르게 수행해야 모델이 데이터를 정확하게 이해하고, 좋은 성능을 낼 수 있음.
- 이번 실습에서는 와인 데이터를 토대로, 이진 분류를 진행해 볼 예정

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine


# 데이터 불러오기
# as_frame=True: pandas DataFrame 형태로 반환
# return_X_y=True: feature와 target을 분리하여 반환
df, y = load_wine(as_frame=True, return_X_y=True)

# label 종류 출력
print(f'y 종류: {y.unique()}')

# 이진과제를 수행할 예정이므로 class 1과 2를 합쳐 1로 변경 할 것
# class 0은 0으로 유지
# class 1과 2는 1로 변경
y[y == 0] = 0
y[y != 0] = 1

# feature와 target을 하나의 DataFrame으로 합치기
df['quality'] = y

# label 확인
label = df['quality'].value_counts().sort_index()
print(f'\n라벨 분포:\n{label}')


## 2-2. 목표
1. `train_test_spilt` 을 사용하여 데이터를 학습용과 테스트용으로 분리
2. `StandardScaler`를 사용하여 데이터를 표준화

## 2-3. 단계별 실습 진행

1. 데이터 분할 (train_test_split)
  - `test_size`: 테스트 데이터의 비율을 설정
    - `0.3`으로 설정시 전체 데이터의 30%를 테스트용으로 사용
  - `random_state`: seed 설정 방식과 동일
  - `stratify`: 데이터셋에 있는 클래스(여기서는 y의 값)의 비율을 `train`과 `test` 데이터셋에 동일하게 맞춰주는 기능
    - 만약, stratify를 설정하지 않으면, 학습용 혹은 테스트용 둘 중, 한곳에만 클래스가 몰려버리는 문제가 발생 할 수 있음.
  


In [ ]:
# model_selection: 데이터 분할과 관련된 다양한 기능들이 포함되어 있음
from sklearn.model_selection import train_test_split

# 1. Train/test 데이터 나누기
    # X: feature 데이터, y: target 데이터
    # `sklearn.model_selection.train_test_split`를 사용
        # 테스트 데이터: 30%로 설정하기
        # 결과 재현을 위해 난수 시드 고정하기
        # 데이터셋 클래스 비율 유지하기
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.3, random_state=42, stratify=y
)

print(f'와인 데이터 shape: {df.shape}')
print(f'훈련 데이터 shape: {X_train.shape}, 테스트 데이터 shape: {X_test.shape}')

2. 데이터 표준화 (StrandardScaler)
> `StrandardScaler`의 역할은, 
> 데이터의 스케일을 조정해서 모든 특성(feature)가 동일한 중요성을 가지도록 만듦

- 예) 키, 몸무게, 나이와 같은 특성들은 각기 다른 단위를 가짐.
- 만약 이 데이터를 그대로 모델에 넣으면, 모델은 단순히 "숫자가 크다는 이유"만으로 키를 몸무게나 나이보다 훨씬 중요한 정보로 인식할 수 있음.
- 이러한 문제를 해결하기 위해, 모든 특성 데이터의 평균을 0으로, 표준편차를 1로 맞추는 작업을 진행

  - 표준화 방법 ($z=x−μ/σ$)

      - $x$: 변환하려는 데이터의 원래 값
      - $μ$: 그 데이터 특성의 평균
      - $σ$: 그 데이터 특성의 표준 편차
      - $z$: 변환된 값(Z-Score)
          - 원래 데이터가 평균으로부터 얼마나 떨어져 있는지를 표준편차 단위로 나타냄.
          - 예를들어, 전체 평균 키가 170cm, 표준편차가 5cm일때, 어떤 사람의 키가 180cm라면 Z-Score는 $z = (180 - 170) / 5 = 2$
    
---

  - `.fit`: `X_train` 데이터만으로 평균과 표준편차를 학습
      - 절대로 `X_test` 데이터는 학습에 사용하지 않음
      - 테스트 데이터로 학습을 수행하는 경우, 데이터 누수(Data Leakage)가 발생 할 수 있음. 
          - 데이터 누수
              - 학습과정에서 테스트 데이터셋에 대한 정보가 모델에 유입되는 현상. 
              - 미리 평가 대상 결과를 학습하게되어 올바른 평가가 진행 될 수 없게 됨.
  - `.transform`: `fit` 단계에서 학습한 평균/표준편차를 사용해 `X_train`과 `X_test`를 모두 변환
      - `X_train` 기준으로 `X_test`를 변환해야 공정한 평가가 가능

In [ ]:
# preprocessing: 데이터 전처리와 관련된 다양한 기능들이 포함되어 있음
from sklearn.preprocessing import StandardScaler
# 2. 표준화 진행
scaler = StandardScaler()

# 진행 순서에 유의
# 1) 훈련 데이터로부터 표준화에 필요한 통계값(평균, 표준편차) 계산
# 2) 훈련 데이터에 대해 표준화 적용
# 3) 테스트 데이터에 대해 표준화 적용(1번에서 계산된 통계값 사용)
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

print(f'표준화된 훈련 데이터: \n {X_train_norm}')

3. 비교
    - `fit_transform()`을 통해 훈련 데이터의 평균/표준편차를 학습하고, 표준화를 진행
    - `transform()` 과정에서는 `.fit()`을 수행하지 않고, 이전에 학습했던 훈련 데이터의 평균/표준편라 정보를 토대로 테스트용 데이터를 변환

In [ ]:
# 3. 비교
# 스케일러의 평균과 표준편차, 훈련/테스트 데이터의 평균과 표준편차 비교
# 서로 비슷한지 확인

print('스케일러의 평균과 표준편차')
print(f'평균: \n{scaler.mean_}')
print(f'표준편차: \n{np.sqrt(scaler.var_)}')

print('\n테스트 데이터의 평균과 표준편차')
print(f'평균: \n{X_test.mean(axis=0).values}')
print(f'표준편차: \n{X_test.std(axis=0).values}')

# 3. 실습: 모델 훈련 및 검증
## 3-1. 들어가기 전에

### 3-1-1. 분류(Classification)

- 이번 실습에서는 “이 와인이 **좋은 와인인가, 아닌가?**”를 분류 할 것.
    - 따라서, quality 값을 0과 1 두 가지 값으로 바꾸는 작업이 필요하고,
    - 이후 예측 결과도 0 혹은 1로 귀결될 것.
    
- 로지스틱 회귀(Logistic Regression)란? 지도학습의 한 종류
    - 어떤 데이터가 두 개의 그룹 (예: 합격/불합격, 정상/비정상) 중 어디에 속하는지를 예측하는 모델 (이진 분류)
    - 분류 방법은 확률을 이용하여 진행.
        - 합격/불합격을 분류하고자 한다면, 투입한 데이터의 특성 (공부 시간, 복습 횟수 등)을 바탕으로 "합격할 확률"을 계산.
        - 계산된 확률을 미리 정해둔 기준점(일반적으로 50%)를 넘으면 합격 / 아니면 불합격으로 판정
        - 예: 어떤 학생이 시험에 합격할 확률이 80%라고 예측하면, 모델은 "합격" 이라는 결론을 내림

### 3-1-2. 회귀(Regression)와 비교
 
- **통계학에서의 회귀:** 어떤 결과가 평균으로 돌아가려는 경향을 의미
- **머신러닝에서의 회귀:** 주어진 데이터의 `특성(feature)`들을 기반으로, 아직 알 수 없는 `연속적인 값(label)`을 `예측`하는 것
    - 예를 들어
        1. 집의 크기, 방의 개수, 위치를 가지고 집값을 예측하는 문제
        2. 기온, 습도, 풍속을 가지고 내일의 전력 사용량을 예측하는 문제
        3. 공부 시간, 복습 횟수를 가지고 시험 점수를 예측하는 문제
    
## 3-2. 목표
  1. 전처리한 데이터를 토대로 `로지스틱 회귀` 모델을 학습
  2. 모델이 얼마나 잘 학습되었는지 평가




## 3-3. 단계별 실습 진행

### 3-3-1. 사이킬럿 3단계 진행
1. 모델 준비 및 Instance 선언
2. fit 메서드를 사용해 모델에게 훈련용 데이터의 특성(feature)와 정답(label)을 학습
3. predict 메서드를 사용해 테스트용 데이터의 예측값 저장

- `ConvergenceWarning`이 발생하였다면?
    - ConvergenceWarning은 모델이 주어진 반복(iteration) 내에 수렴하지 못했음을 의미
        - 즉, 모델이 최적의 파라미터를 찾지 못했음을 의미
        - 해결 방법
            1. 더 많은 반복(iteration) 허용: `max_iter` 파라미터 증가
            2. 데이터 스케일링: `StandardScaler` 등으로 데이터 전처리
            3. 다른 최적화 알고리즘 사용: `solver` 파라미터 변경
            4. 모델 단순화: 불필요한 특성 제거
    - [참고](https://www.slingacademy.com/article/understanding-scikit-learns-convergencewarning-and-how-to-resolve-it/)

In [ ]:
# linear_model: 선형 모델과 관련된 다양한 기능들이 포함되어 있음
    # 로지스틱 회귀, 선형 회귀 등
from sklearn.linear_model import LogisticRegression

# 1. LogisticRegression 모델을 선언해 clf에 할당
    # ConvergenceWarning이 발생하였다면, max_iter 값을 늘려서 해결 시도
    # clf = LogisticRegression()
clf = LogisticRegression(max_iter=3000)

# 2. .fit을 통해 모델을 학습
clf.fit(X_train, y_train)

# 3. X_test를 예측해서 y_pred에 할당
y_pred = clf.predict(X_test)
# 테스트 데이터에 대한 예측 결과 출력 (와인 등급이 0인지 1인지)
print(y_pred)

### 3-3-2. 평가

- 분류(Classification)에서의 분류는 어떤 데이터가 `어떤 그룹에 속하는가?` 를 예측하는 문제.
- 예측이 맞았는지, 틀렸는지를 기준으로 평가 지표를 만들어서 판단

1. **혼동 행렬(Confusion Matrix):** 가장 기본적인 평가 방법
    
    
    |  | 모델 예측: 양성 (합격) | 모델 예측: 음성 (불합격) |
    | --- | --- | --- |
    | 실제 정답: 양성 (합격) | True Positive (진짜 양성) | False Negative (가짜 음성) |
    | 실제 정답: 음성 (불합격) | False Positive (가짜 양성) | True Negative (진짜 음성) |

1. 평가 지표의 종류
    1. **정확도**(Accuracy)**:** 전체 예측 중, 모델이 정답을 맞힌 비율
        - $Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$
    2. **정밀도**(Precision)**:** 모델이 정답이라고 예측한 것들**(진짜 양성 + 가짜 양성)**중, 진짜 정답인 비율
        - $Precision = \frac {TP}{TP + FP}$
    3. **재현율**(Recall)**:** 진짜 정답인 것들 중, 모델이 정답이라고 **올바르게** 예측한 비율
        - $Recall = \frac{TP}{TP + FN}$
    4. **F1-Score**: 정밀도와 재현율의 균형을 나타냄. 둘 중 하나만 높고 나머지는 낮은지를 판별
        - $F1-Score = 2 * {(정밀도 * 재현율) \over (정밀도 + 재현율)}$


In [ ]:
# metrics: 모델 평가와 관련된 다양한 기능들이 포함되어 있음
from sklearn.metrics import confusion_matrix, classification_report
# 혼동 행렬 (실제 정답, 예측 결과 비교)
print(confusion_matrix(y_test, y_pred))

print("================================")
# 분류 보고서 (정밀도, 재현율, F1-Score 등)
print(classification_report(y_test, y_pred))

### 3-3-3. ROC-AUC
- 모델의 분류 성능을 종합적으로 평가
- **ROC 곡선**
    - 여러 임계값(Threshold)에 따라 **재현율**과 **위양성율**의 관계를 시각적으로 보여주는 그래프
- **AUC(Area Under Curve)**
    - ROC 곡선 아래의 넓이를 의미
    - AUC 값이 클수록(1에 가까울수록) 모델의 성능이 좋다는 뜻
    - **0.5는 무작위 예측과 같다는 의미라, 0.5보다 높아야 의미가 있음.**

In [ ]:

# metrics: 모델 평가와 관련된 다양한 기능들이 포함되어 있음
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# 양성(1) 클래스의 "점수" 얻기: 가능하면 확률, 없으면 decision_function 점수
y_score = clf.predict_proba(X_test)[:, 1]

# FPR, TPR, 임계값과 AUC 계산
fpr, tpr, thresholds = roc_curve(y_test, y_score)
auc = roc_auc_score(y_test, y_score)

# ROC 곡선 시각화 (matplotlib 사용)
plt.plot(fpr, tpr, label=f'ROC-AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

# 4. 실습: 교차 검증 (Cross-Validation)

## 4-1. 교차검증이란?

- 데이터를 여러 번 다르게 나눠서 학습과 평가를 반복하는 방법
- 모델의 성능을 한 번의 시험으로 판단하지 않고, 여러 번의 시험 점수를 평균 내서 더 믿을 수 있는 실력을 평가
- 가장 흔하게 사용되는 방법은  **K-Fold 교차 검증**

## 4-2. 다양한 교차 검증 방법

1. **K-Fold**
    - 전체 학습 데이터 (`X_Train`)을 똑같은 크기의 K개의 조각으로 나누는 방식
    - 우연한 편차를 줄이고 모델의 `일반화 성능`을 더 안정적으로 평가 할 수 있음.
    1. 첫 번째 Fold를 검증 데이터로 사용하고, 나머지 K-1개 Fold는 학습 데이터로 사용해 모델을 학습하고 평가
    2. 위 평가가 종료된 후, 다시 두 번째 Fold를 검증 데이터로 사용하고, 나머지는 학습 데이터로 쓰는 과정을 반복
    3. 이 과정을 K번 반복하면, 모든 Fold가 한 번씩 검증 데이터가 되는 방식
    4. 마지막으로 K번의 평가 끝에 얻은 점수들을 평균하여 최종 점수를 얻음.
2. **층화 K-Fold (Stratifield K-Fold)**
    - **데이터가 불균형 할 때,** K**-**Fold보다 효과적
    - 원래 데이터 셋의 클래스 비율을 항상 각 Fold에서 항상 동일하게 유지하도록 데이터를 나누어서 진행
    - 진행 방식은 K-Fold와 동일
3. **Leave-One-Out Cross-Validation (LOOCV)**
    - K-Fold의 극단적인 형태
    - 전체 데이터의 개수 ($N$)개 만큼 폴드를 생성 ($K = N$)
        - 매 검증마다 **단 하나의 데이터**만 검증 데이터로 사용하고, 나머지 모든 데이터를 학습에 사용
        - 이 과정을 N번 반복
4. **그룹 K-Fold (Group K-Fold)**
    - 데이터에 **서로 종속적인 그룹**이 있을 때 유용
    - 같은 그룹에 속하는 데이터 (예 - 한 환자에게서 얻은 모든 데이터)가 항상 같은 Fold에 있도록 처리.
        - 이를 통해, 모델이 학습 단계에서 본 데이터를, 검증 단계에서 다시 보는 데이터 누수를 막을 수 있음

In [ ]:
# model_selection: 데이터 분할과 관련된 다양한 기능들이 포함되어 있음
from sklearn.model_selection import cross_val_score

# k-fold 교차검증 진행하기
# k-fold 교차검증 진행
    # 현재 선택한 하이퍼파라미터가 데이터 분할에 따라 편향되지 않고,
    # 안정적으로 일반화 성능을 내는지 확인하는 과정
# 각 fold마다 모델을 새로 학습(fit) 후 검증 데이터를 평가하며,
# 최종적으로 평균 F1-score를 산출

# estimator: 평가할 모델
    # clf는 앞서 정의한 LogisticRegression 모델
f1_scores = cross_val_score(estimator=clf, X=X_train, y=y_train, cv=5, scoring='f1')
print("Average F1-score (CV):", f1_scores.mean())